# Chronos-2 (120M): M6 quantile inference, rounds 1 to 12

Zero-shot inference for all 12 official M6 rounds. Chronos-2 differs from the
other two models in what it returns: instead of sampled trajectories it outputs
21 quantile levels per asset and day, so that is what gets saved here.

| | |
|---|---|
| Model | `amazon/chronos-2`, zero-shot, no fine-tuning |
| Output form | 21 quantile levels per asset and forecast day |
| Fixed | context 512 weekdays, horizon 20, seed 42, official M6 asset order |
| Input | the 12 round-context CSVs on Drive |
| Output | 12 quantile NPZ files |

Because there are no sampled paths, scenarios have to be reconstructed from
these quantiles before the M6 quintile probabilities can be formed. That is done
later by the Chronos-2 post-processing scripts, not here.

## 2. Install Chronos-2

`chronos-forecasting==2.3.1` pinned to match the version used for the earlier two
models, and Transformers held on a 4.x release, Transformers 5.x is not
compatible with this Chronos release.

Colab may ask you to restart the runtime after installing. If it does, restart
and then re-run from this cell.

In [ ]:
%pip install -q "chronos-forecasting==2.3.1" "transformers>=4.41,<5"

## 3. Imports, versions and settings

Prints the exact package versions for reproducibility, then defines the settings.
Note what is absent compared with the earlier notebooks: no `NUM_SAMPLES` and no
`SERIES_BATCH_SIZE`, because Chronos-2 returns quantiles and takes all 100 assets
as one task.

In [ ]:
import traceback
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from chronos import Chronos2Pipeline

# --- Model ------------------------------------------------------------------
MODEL_ID = "amazon/chronos-2"
MODEL_SIZE = "~120M parameters"

# --- Experiment settings ----------------------------------------------------
CONTEXT_LENGTH = 512
PREDICTION_LENGTH = 20
RANDOM_SEED = 42          # set for reproducibility; quantile output is deterministic
ROUNDS = range(1, 13)

# Chronos-2 shares information across the variates of ONE task. cross_learning
# shares information across SEPARATE tasks - we have a single task, so it stays
# False (the API default).
CROSS_LEARNING = False


# Documented native quantile levels; the actual levels are read from the loaded
# pipeline in section 8 and these are used only to check them.
EXPECTED_QUANTILE_LEVELS = (
    [0.01, 0.05] + [round(0.05 * i, 2) for i in range(2, 19)] + [0.95, 0.99]
)

# --- Official M6 round schedule (context-building step, pre-specified) --------------------
# round -> context start, origin (= context end), forecast start, forecast end
ROUND_SCHEDULE = {
    1:  ("2020-03-19", "2022-03-04", "2022-03-07", "2022-04-01"),
    2:  ("2020-04-16", "2022-04-01", "2022-04-04", "2022-04-29"),
    3:  ("2020-05-14", "2022-04-29", "2022-05-02", "2022-05-27"),
    4:  ("2020-06-11", "2022-05-27", "2022-05-30", "2022-06-24"),
    5:  ("2020-07-09", "2022-06-24", "2022-06-27", "2022-07-22"),
    6:  ("2020-08-06", "2022-07-22", "2022-07-25", "2022-08-19"),
    7:  ("2020-09-03", "2022-08-19", "2022-08-22", "2022-09-16"),
    8:  ("2020-10-01", "2022-09-16", "2022-09-19", "2022-10-14"),
    9:  ("2020-10-29", "2022-10-14", "2022-10-17", "2022-11-11"),
    10: ("2020-11-26", "2022-11-11", "2022-11-14", "2022-12-09"),
    11: ("2020-12-24", "2022-12-09", "2022-12-12", "2023-01-06"),
    12: ("2021-01-21", "2023-01-06", "2023-01-09", "2023-02-03"),
}

# Genuine leading missing history, per context-building step (verified against the repo files).
EXPECTED_LEADING_NAN = {r: {"OGN": 302 - 20 * (r - 1)} for r in ROUNDS}
EXPECTED_LEADING_NAN[1] = {"CARR": 1, "OGN": 302}

# Official M6 asset order - used to verify the contexts, not to reorder them.
OFFICIAL_ASSET_ORDER = [
    "ABBV", "ACN", "AEP", "AIZ", "ALLE", "AMAT", "AMP", "AMZN", "AVB", "AVY",
    "AXP", "BDX", "BF-B", "BMY", "BR", "CARR", "CDW", "CE", "CHTR", "CNC",
    "CNP", "COP", "CTAS", "CZR", "DG", "DPZ", "DRE", "DXC", "EWA", "EWC",
    "EWG", "EWH", "EWJ", "EWL", "EWQ", "EWT", "EWU", "EWY", "EWZ", "FTV",
    "GOOG", "GPC", "GSG", "HIG", "HIGH.L", "HST", "HYG", "IAU", "ICLN",
    "IEAA.L", "IEF", "IEFM.L", "IEMG", "IEUS", "IEVL.L", "IGF", "INDA",
    "IUMO.L", "IUVL.L", "IVV", "IWM", "IXN", "JPEA.L", "JPM", "KR", "LQD",
    "MCHI", "META", "MVEU.L", "OGN", "PG", "PPL", "PRU", "PYPL", "RE",
    "REET", "ROL", "ROST", "SEGA.L", "SHY", "SLV", "SPMV.L", "TLT", "UNH",
    "URI", "V", "VRSK", "VXX", "WRK", "XLB", "XLC", "XLE", "XLF", "XLI",
    "XLK", "XLP", "XLU", "XLV", "XLY", "XOM",
]
N_ASSETS = len(OFFICIAL_ASSET_ORDER)

print("Versions")
print(f"  chronos-forecasting : {version('chronos-forecasting')}")
print(f"  transformers        : {version('transformers')}")
print(f"  torch               : {torch.__version__}")
print(f"  torch CUDA build    : {torch.version.cuda}")
print(f"  CUDA available      : {torch.cuda.is_available()}")
print(f"\nExperiment: {MODEL_ID} ({MODEL_SIZE}), zero-shot, rounds "
      f"{min(ROUNDS)}-{max(ROUNDS)}")
print(f"Model input per round : (1, {N_ASSETS}, {CONTEXT_LENGTH})  "
      "= 1 multivariate task x 100 target variates x 512 observations")
print(f"Saved array per round : ({N_ASSETS}, {PREDICTION_LENGTH}, "
      f"{len(EXPECTED_QUANTILE_LEVELS)})  = assets x forecast days x quantiles")

## 4. Mount Google Drive

Supplies the existing context files and stores the outputs permanently.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 5. Configure paths

`DRIVE_CONTEXT_DIR` is the folder both earlier models already read, the same 12
context CSVs, not another copy.

**`DRIVE_OUTPUT_DIR` is the one path to set before running.** It is created
automatically and nothing is written near the other models' outputs.

In [ ]:
# Existing shared context folder - same files as the earlier model runs.
DRIVE_CONTEXT_DIR = Path("/content/drive/MyDrive/HonoursResearch/contexts")

# >>> SET THIS BEFORE RUNNING - the only path you need to change <<<
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/HonoursResearch/outputs/chronos2/quantiles")

OUTPUT_PREFIX = "chronos_2"


def context_path(round_number: int) -> Path:
    return DRIVE_CONTEXT_DIR / f"round_{round_number:02d}_context.csv"


def quantiles_path(round_number: int) -> Path:
    return DRIVE_OUTPUT_DIR / f"{OUTPUT_PREFIX}_round{round_number:02d}_quantiles.npz"


if not DRIVE_CONTEXT_DIR.is_dir():
    raise FileNotFoundError(
        f"Context folder not found:\n  {DRIVE_CONTEXT_DIR}\n"
        "It must be the SAME folder used by the earlier model notebooks, "
        "containing round_01_context.csv .. round_12_context.csv."
    )

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Contexts (existing, shared): {DRIVE_CONTEXT_DIR}")
print(f"Output folder (Chronos-2)  : {DRIVE_OUTPUT_DIR}")
print(f"Example output filename    : {quantiles_path(1).name}")

## 5. Check the 12 context files are present

In [ ]:
missing = [context_path(r).name for r in ROUNDS if not context_path(r).is_file()]
if missing:
    raise FileNotFoundError(
        f"Missing context files in {DRIVE_CONTEXT_DIR}:\n  " + "\n  ".join(missing)
        + "\n\nCopy round_01_context.csv .. round_12_context.csv there from "
          "Data/contexts/.")

print(f"All {len(list(ROUNDS))} context files present in {DRIVE_CONTEXT_DIR}")

## 7. GPU check

A GPU is required; the cell stops rather than silently running on CPU.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. Runtime > Change runtime type > Hardware "
        "accelerator: GPU, then re-run from section 3."
    )

GPU_NAME = torch.cuda.get_device_name(0)
print(f"GPU        : {GPU_NAME}")
print(f"CUDA (torch): {torch.version.cuda}")

## 8. Load Chronos-2, **once**

> ⚠️ **This downloads/loads `amazon/chronos-2` onto the GPU.** The weights go to
> the Colab runtime cache, never into the repository.

Loaded once and reused for all 12 rounds. Zero-shot inference only: no training,
fine-tuning or weight modification. The cell also reads the model's **native
quantile levels** from the pipeline and checks them against the documented 21
levels rather than imposing a hard-coded list.

In [ ]:
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

pipeline = Chronos2Pipeline.from_pretrained(
    MODEL_ID,
    device_map="cuda",
)

# Native quantile levels, taken from the loaded model.
QUANTILE_LEVELS = np.asarray(pipeline.quantiles, dtype=np.float64)
N_QUANTILES = int(QUANTILE_LEVELS.size)

assert np.all(np.diff(QUANTILE_LEVELS) > 0), "Quantile levels are not increasing"
assert N_QUANTILES == len(EXPECTED_QUANTILE_LEVELS), (
    f"Expected {len(EXPECTED_QUANTILE_LEVELS)} native quantile levels, "
    f"the loaded model reports {N_QUANTILES}: {QUANTILE_LEVELS.tolist()}"
)
if not np.allclose(QUANTILE_LEVELS, EXPECTED_QUANTILE_LEVELS):
    print("NOTE: the model's quantile levels differ from the documented set; "
          "the model's own levels are used and saved.")

n_params = sum(p.numel() for p in pipeline.model.parameters())
MODEL_LOADED_ONCE = True

print(f"Loaded: {MODEL_ID}")
print(f"  parameters      : {n_params:,} ({n_params / 1e6:.1f}M)")
print(f"  device          : {GPU_NAME}")
print(f"  quantile levels : {N_QUANTILES} -> {QUANTILE_LEVELS.tolist()}")
print("  zero-shot inference only - no training or fine-tuning.")

## 9. Rounds 1-12

### 9a. Helpers

* `load_and_validate_context`, identical to the earlier notebooks: 512 rows,
  `date` plus the 100 official assets in order, ascending unique dates, the
  round's official context start and origin, the correct 20-weekday forecast
  window, and the expected genuine leading `NaN`s with no interior gaps. It
  returns the `(100 assets, 512 steps)` matrix and proves it is a plain transpose
  of the loaded values.
* `forecast_round`, builds the `(1, 100, 512)` multivariate task, calls
  `pipeline.predict(...)`, and normalises the result to
  `(100 assets, 20 days, 21 quantiles)`. The API documents its return as
  `(n_variates, n_quantiles, prediction_length)`, but rather than trusting that,
  the function *inspects* the returned shape and transposes only if needed -
  20 days and 21 quantiles are different sizes, so the axes are unambiguous.
* `save_and_verify_round`, writes the NPZ, then reloads it and checks the four
  arrays, the shape, asset ordering, forecast dates, quantile levels, finiteness
  and quantile monotonicity.

In [ ]:
def load_and_validate_context(round_number: int):
    """Load one model context, validate it, and build the model input."""
    ctx_start, origin, fc_start, fc_end = ROUND_SCHEDULE[round_number]
    df = pd.read_csv(context_path(round_number), parse_dates=["date"])

    assert df.shape[0] == CONTEXT_LENGTH, (
        f"Round {round_number}: expected {CONTEXT_LENGTH} rows, found {df.shape[0]}"
    )
    assert list(df.columns) == ["date"] + OFFICIAL_ASSET_ORDER, (
        f"Round {round_number}: columns are not 'date' + the official M6 asset order"
    )
    dates = df["date"]
    assert dates.is_monotonic_increasing, f"Round {round_number}: dates not ascending"
    assert not dates.duplicated().any(), f"Round {round_number}: duplicate dates"
    assert dates.iloc[0] == pd.Timestamp(ctx_start), f"Round {round_number}: context start"
    assert dates.iloc[-1] == pd.Timestamp(origin), f"Round {round_number}: context end/origin"
    assert (dates <= pd.Timestamp(origin)).all(), (
        f"Round {round_number}: context contains a date after the origin"
    )

    matrix = df[OFFICIAL_ASSET_ORDER].to_numpy(dtype=np.float64).T
    assert matrix.shape == (N_ASSETS, CONTEXT_LENGTH)
    assert np.array_equal(
        matrix, df[OFFICIAL_ASSET_ORDER].to_numpy().T, equal_nan=True
    ), f"Round {round_number}: model input differs from the loaded context values"

    leading_missing, interior_missing = {}, {}
    for i, symbol in enumerate(OFFICIAL_ASSET_ORDER):
        row = matrix[i]
        n_lead = int(np.argmax(~np.isnan(row))) if np.isnan(row[0]) else 0
        if n_lead:
            leading_missing[symbol] = n_lead
        if np.isnan(row[n_lead:]).any():
            interior_missing[symbol] = int(np.isnan(row[n_lead:]).sum())
    assert not interior_missing, (
        f"Round {round_number}: unexpected interior NaNs {interior_missing}"
    )
    assert leading_missing == EXPECTED_LEADING_NAN[round_number], (
        f"Round {round_number}: leading NaN pattern changed - "
        f"expected {EXPECTED_LEADING_NAN[round_number]}, found {leading_missing}"
    )

    forecast_dates = pd.bdate_range(fc_start, fc_end)
    assert len(forecast_dates) == PREDICTION_LENGTH, f"Round {round_number}: forecast window"
    return matrix, df, forecast_dates


def forecast_round(context_matrix, round_number: int):
    """Forecast one round as a single multivariate task.

    Input  : (1, 100, 512) - one task, 100 target variates, 512 observations.
    Output : (100, 20, 21) - assets x forecast days x native quantiles.

    The only change made to the context values is the float32 cast the tensor
    needs; NaNs are passed through for Chronos-2's own missing-value handling.
    """
    model_input = context_matrix[np.newaxis, :, :].astype(np.float32)
    assert model_input.shape == (1, N_ASSETS, CONTEXT_LENGTH), (
        f"Round {round_number}: model input {model_input.shape}, "
        f"expected (1, {N_ASSETS}, {CONTEXT_LENGTH})"
    )

    outputs = pipeline.predict(
        inputs=model_input,
        prediction_length=PREDICTION_LENGTH,
        cross_learning=CROSS_LEARNING,
    )
    assert len(outputs) == 1, (
        f"Round {round_number}: expected 1 task in the output, got {len(outputs)}"
    )
    raw = outputs[0]
    raw = raw.to(torch.float32).cpu().numpy() if torch.is_tensor(raw) else np.asarray(raw)

    # Inspect the returned layout instead of assuming it. 20 days and 21
    # quantiles differ in size, so the axes can be identified unambiguously.
    if raw.shape == (N_ASSETS, N_QUANTILES, PREDICTION_LENGTH):
        quantile_forecasts = np.transpose(raw, (0, 2, 1))   # -> assets, days, quantiles
        layout = "(assets, quantiles, days) -> transposed"
    elif raw.shape == (N_ASSETS, PREDICTION_LENGTH, N_QUANTILES):
        quantile_forecasts = raw
        layout = "(assets, days, quantiles) -> already correct"
    else:
        raise ValueError(
            f"Round {round_number}: unrecognised output shape {raw.shape}; expected a "
            f"permutation of ({N_ASSETS}, {PREDICTION_LENGTH}, {N_QUANTILES})"
        )

    quantile_forecasts = np.ascontiguousarray(quantile_forecasts)
    assert quantile_forecasts.shape == (N_ASSETS, PREDICTION_LENGTH, N_QUANTILES)
    assert np.isfinite(quantile_forecasts).all(), (
        f"Round {round_number}: forecasts contain NaN or infinite values"
    )

    # Quantiles must not cross, for every asset and every forecast day.
    max_violation = float(-np.min(np.diff(quantile_forecasts, axis=2)))
    assert max_violation <= 1e-6, (
        f"Round {round_number}: quantiles are not monotonically ordered "
        f"(largest crossing {max_violation:.3e})"
    )

    return quantile_forecasts, layout, max_violation


def save_and_verify_round(round_number: int, quantile_forecasts, forecast_dates):
    """Save the raw quantile forecasts, then reload and verify the file."""
    path = quantiles_path(round_number)
    date_strings = np.array([d.strftime("%Y-%m-%d") for d in forecast_dates])

    np.savez_compressed(
        path,
        quantile_forecasts=quantile_forecasts,
        quantile_levels=QUANTILE_LEVELS,
        asset_symbols=np.array(OFFICIAL_ASSET_ORDER),
        forecast_dates=date_strings,
    )

    with np.load(path, allow_pickle=False) as reloaded:
        assert set(reloaded.files) == {
            "quantile_forecasts", "quantile_levels", "asset_symbols", "forecast_dates"
        }, f"Round {round_number}: unexpected arrays {sorted(reloaded.files)}"
        r_forecasts = reloaded["quantile_forecasts"]
        r_levels = reloaded["quantile_levels"]
        r_symbols = reloaded["asset_symbols"]
        r_dates = reloaded["forecast_dates"]

    assert r_forecasts.shape == (N_ASSETS, PREDICTION_LENGTH, N_QUANTILES), "reloaded shape"
    assert np.array_equal(r_forecasts, quantile_forecasts), "saved values differ"
    assert np.isfinite(r_forecasts).all(), "reloaded values not finite"
    assert np.allclose(r_levels, QUANTILE_LEVELS), "quantile levels changed on save/reload"
    assert list(r_symbols) == OFFICIAL_ASSET_ORDER, "asset ordering changed on save/reload"
    assert list(r_dates) == list(date_strings), "forecast dates changed on save/reload"
    return path


print("Helpers defined.")

### 9b. Run Rounds 1-12

One multivariate forecast per round, saved and re-verified immediately, so a
failure in a later round cannot lose earlier work. A failed round is recorded and
the loop continues.

If the GPU runs out of memory, note that the 100 assets **cannot** simply be
split into batches, that would break the single joint task this stage is
testing. Reduce memory some other way (a larger GPU) rather than by splitting.

In [ ]:
records, failures = [], []
run_started_utc = datetime.now(timezone.utc)

for r in ROUNDS:
    _, origin, fc_start, fc_end = ROUND_SCHEDULE[r]
    print(f"\n=== Round {r:02d} | origin {origin} | forecast {fc_start} .. {fc_end} ===")
    try:
        context_matrix, context_df, forecast_dates = load_and_validate_context(r)
        print(f"  context validated: {context_matrix.shape} (assets x time steps) -> "
              f"task input (1, {N_ASSETS}, {CONTEXT_LENGTH})")

        started = datetime.now(timezone.utc)
        quantile_forecasts, layout, max_violation = forecast_round(context_matrix, r)
        elapsed = (datetime.now(timezone.utc) - started).total_seconds()

        saved_path = save_and_verify_round(r, quantile_forecasts, forecast_dates)
        records.append({
            "round": r, "origin": origin,
            "forecast_start": fc_start, "forecast_end": fc_end,
            "shape": tuple(quantile_forecasts.shape),
            "seconds": round(elapsed, 1), "file": saved_path.name,
        })
        print(f"  output {tuple(quantile_forecasts.shape)} | api layout {layout}")
        print(f"  all finite, quantiles monotonic (max crossing {max_violation:.2e}) "
              f"| {elapsed:.1f}s")
        print(f"  saved and verified -> {saved_path.name}")

    except Exception as exc:
        failures.append((r, f"{type(exc).__name__}: {exc}"))
        print(f"  !! ROUND {r:02d} FAILED - earlier rounds remain saved")
        traceback.print_exc()

print(f"\nCompleted rounds: {[rec['round'] for rec in records]}")
if failures:
    print(f"FAILED rounds   : {[r for r, _ in failures]} (rerun these)")
else:
    print("No failures.")

## 10. Verify all 12 saved files

An independent pass over what is actually on Drive, nothing here relies on the
in-memory arrays.

In [ ]:
ALL_VERIFIED = True
print(f"{'Rnd':>3}  {'file':<38} {'status':<7} {'shape':<18} {'order':<6} {'dates':<6} finite")

for r in ROUNDS:
    path = quantiles_path(r)
    if not path.is_file():
        ALL_VERIFIED = False
        print(f"{r:>3}  {path.name:<38} MISSING")
        continue

    _, _, fc_start, fc_end = ROUND_SCHEDULE[r]
    expected_dates = [d.strftime("%Y-%m-%d") for d in pd.bdate_range(fc_start, fc_end)]
    with np.load(path, allow_pickle=False) as f:
        keys = set(f.files)
        forecasts = f["quantile_forecasts"]
        levels = f["quantile_levels"]
        symbols = list(f["asset_symbols"])
        dates = list(f["forecast_dates"])

    ok_keys = keys == {"quantile_forecasts", "quantile_levels", "asset_symbols", "forecast_dates"}
    ok_shape = forecasts.shape == (N_ASSETS, PREDICTION_LENGTH, N_QUANTILES)
    ok_levels = levels.size == N_QUANTILES and np.allclose(levels, QUANTILE_LEVELS)
    ok_order = symbols == OFFICIAL_ASSET_ORDER
    ok_dates = dates == expected_dates
    ok_finite = bool(np.isfinite(forecasts).all())
    ok_monotonic = ok_shape and float(-np.min(np.diff(forecasts, axis=2))) <= 1e-6
    ok = all([ok_keys, ok_shape, ok_levels, ok_order, ok_dates, ok_finite, ok_monotonic])
    ALL_VERIFIED = ALL_VERIFIED and ok

    print(f"{r:>3}  {path.name:<38} {'OK' if ok else 'FAILED':<7} {str(forecasts.shape):<18} "
          f"{'yes' if ok_order else 'NO':<6} {'yes' if ok_dates else 'NO':<6} "
          f"{'yes' if ok_finite and ok_monotonic else 'NO'}")

print(f"\nAll 12 rounds verified on Drive: {ALL_VERIFIED}")

## 11. Completion summary

What to copy back into the repository once the run finishes.

In [ ]:
print("Chronos-2 (120M) - M6 Rounds 1-12, zero-shot")
print(f"  model            : {MODEL_ID} ({MODEL_SIZE}), loaded once, no fine-tuning")
print(f"  task per round   : (1, {N_ASSETS}, {CONTEXT_LENGTH}) - one multivariate task, "
      "all 100 assets as targets")
print(f"  cross_learning   : {CROSS_LEARNING} (single task; variates are already joint)")
print(f"  rounds completed : {len(records)}/{len(list(ROUNDS))} "
      f"-> {[rec['round'] for rec in records]}")
if failures:
    print(f"  rounds FAILED    : {[r for r, _ in failures]}")
print(f"  output per round : ({N_ASSETS}, {PREDICTION_LENGTH}, {N_QUANTILES}) "
      "= assets x forecast days x native quantiles")
print(f"  quantile levels  : {N_QUANTILES} native levels, saved with every file")
print(f"  saved to         : {DRIVE_OUTPUT_DIR}")
print(f"  all verified     : {ALL_VERIFIED}")
print("  post-processing  : none (no four-week returns, quintiles, RPS, DRE change,")
print("                     and no conversion of quantiles into sampled trajectories)")
print("\nCopy the 12 NPZ files into the repository at:")
print("  TSFM Inference and Sampling/inference_outputs/Chronos_2_120M/")